In [ ]:
import os
import zipfile
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Runs both on Google Colab (Drive) and locally (Jupyter / VS Code)
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("Running on", "Google Colab" if IN_COLAB else "local machine")

In [ ]:
if IN_COLAB:
    drive.mount('/content/drive')
    project_path = "/content/drive/MyDrive/DIP"
    dataset_path = os.path.join(project_path, "Dataset")
else:
    project_path = os.getcwd()                       # the folder this notebook is in
    dataset_path = os.path.join(project_path, "data", "Dataset")

output_path = os.path.join(project_path, "Output")
zip_path = os.path.join(dataset_path, "dataset.zip")

os.makedirs(dataset_path, exist_ok=True)
os.makedirs(output_path, exist_ok=True)
print("Dataset:", dataset_path)
print("Output: ", output_path)

In [ ]:
signatures_dir = os.path.join(dataset_path, "signatures")

if os.path.isdir(signatures_dir):
    print("Dataset already extracted, skipping extraction.")
elif os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(dataset_path)
    print("Dataset extracted successfully.")
else:
    print("ZIP file not found at", zip_path)

In [ ]:
import os

for root, dirs, files in os.walk(dataset_path):
    print(root)
    print(files[:5])      # first 5 files

In [ ]:
subfolders = ["Stage1_Contrast", "Stage2_Median", "Stage3_HighBoost", "Stage4_Sobel"]

for folder in subfolders:
    os.makedirs(os.path.join(output_path, folder), exist_ok=True)

print("Output stage directories initialized in", output_path)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

In [ ]:
image_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')
image_files = []

for root, dirs, files in os.walk(dataset_path):
    for file in files:
        if file.lower().endswith(image_extensions):
            image_files.append(os.path.join(root, file))

print(f"Total dataset images found: {len(image_files)}")

In [ ]:
# 1. Contrast Stretching

def contrast_stretching(img):
    """Linear stretch: maps [min, max] -> [0, 255]"""
    r_min, r_max = float(np.min(img)), float(np.max(img))
    if r_max == r_min:
        return img.copy()
    stretched = (img.astype(np.float64) - r_min) * (255.0 / (r_max - r_min))
    return np.uint8(np.clip(stretched, 0, 255))

In [ ]:
# 2. Median Filter

def median_filter(img, ksize=3):
    return cv2.medianBlur(img, ksize)

In [ ]:
# 11. Minimum Filter

def minimum_filter(img, ksize=3):
    kernel = np.ones((ksize, ksize), np.uint8)
    return cv2.erode(img, kernel)

# 14. High-Boost Filtering

def high_boost_filter(img, ksize=3, k=1.5):
    """High-boost = original + k * (original - blurred),  k >= 1"""
    blurred = cv2.blur(img, (ksize, ksize)).astype(np.float64)
    mask = img.astype(np.float64) - blurred
    boosted = img.astype(np.float64) + k * mask
    return np.uint8(np.clip(boosted, 0, 255))

In [ ]:
# 13. Laplacian Filtering (L1: 4-neighbor mask, L2: 8-neighbor mask)

def laplacian_filter(img, mask_type="L1", sharpen=True):
    if mask_type == "L1":
        kernel = np.array([[0, 1, 0],
                            [1, -4, 1],
                            [0, 1, 0]], dtype=np.float64)
    else:  # L2 (diagonals included)
        kernel = np.array([[1, 1, 1],
                            [1, -8, 1],
                            [1, 1, 1]], dtype=np.float64)

    lap = cv2.filter2D(img.astype(np.float64), -1, kernel)
    edge_map = np.uint8(np.clip(np.abs(lap), 0, 255))

    if not sharpen:
        return edge_map

    sharpened = img.astype(np.float64) - lap  # subtract since center coeff is negative
    sharpened = np.uint8(np.clip(sharpened, 0, 255))
    return edge_map, sharpened

# 15. Sobel Edge Detection

def sobel_edge_detection(img, ksize=3):
    gx = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=ksize)
    gy = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=ksize)
    magnitude = np.sqrt(gx ** 2 + gy ** 2)
    return np.uint8(np.clip(magnitude, 0, 255))

In [ ]:
print("\nStarting pipeline execution...")

for index, image_path in enumerate(image_files):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        print(f"Could not read: {image_path}")
        continue

    # Get path relative to dataset_path to preserve subfolder directory structures
    rel_path = os.path.relpath(image_path, dataset_path)

    # Execute 4 Sequential Stages
    stage1 = contrast_stretching(img)
    stage2 = median_filter(stage1, ksize=3)
    stage3 = high_boost_filter(stage2, ksize=3, k=1.5)
    stage4 = sobel_edge_detection(stage3, ksize=3)

    # Ensure output nested folders exist before saving
    for folder in subfolders:
        save_dir = os.path.dirname(os.path.join(output_path, folder, rel_path))
        os.makedirs(save_dir, exist_ok=True)

    # Save stage results
    cv2.imwrite(os.path.join(output_path, "Stage1_Contrast", rel_path), stage1)
    cv2.imwrite(os.path.join(output_path, "Stage2_Median", rel_path), stage2)
    cv2.imwrite(os.path.join(output_path, "Stage3_HighBoost", rel_path), stage3)
    cv2.imwrite(os.path.join(output_path, "Stage4_Sobel", rel_path), stage4)

    # Progress logging every 50 images
    if (index + 1) % 50 == 0 or (index + 1) == len(image_files):
        print(f"Processed {index + 1}/{len(image_files)} images...")

print("\nProcessing finished! All results saved in", output_path)